# Data Preprocessing

In this notebook, the cleaned ILINet dataset is preprocessed to prepare it for exploratory analysis and forecasting. The preprocessing workflow includes correcting data types, creating temporal features, organizing the dataset for time-series analysis, and exporting the processed dataset for subsequent notebooks.

#### 1. Import Libraries

In [ ]:
# Import libraries
import warnings

import numpy as np
import pandas as pd
from epiweeks import Week

from src.data_loader import load_cleaned_ilinet

warnings.filterwarnings("ignore")

#### 2. Display Settings

In [2]:
# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)
pd.set_option("display.width", 120)

#### 3. Load Dataset

In [3]:
# Load cleaned dataset
df = load_cleaned_ilinet()

#### 4. Dataset Overview

In [4]:
# Display dataset information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1499 entries, 0 to 1498
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   REGION TYPE        1499 non-null   str    
 1   REGION             0 non-null      float64
 2   YEAR               1499 non-null   int64  
 3   WEEK               1499 non-null   int64  
 4   % WEIGHTED ILI     1499 non-null   float64
 5   %UNWEIGHTED ILI    1499 non-null   float64
 6   AGE 0-4            1499 non-null   int64  
 7   AGE 25-49          967 non-null    float64
 8   AGE 25-64          627 non-null    float64
 9   AGE 5-24           1499 non-null   int64  
 10  AGE 50-64          967 non-null    float64
 11  AGE 65             1499 non-null   int64  
 12  ILITOTAL           1499 non-null   int64  
 13  NUM. OF PROVIDERS  1499 non-null   int64  
 14  TOTAL PATIENTS     1499 non-null   int64  
dtypes: float64(6), int64(8), str(1)
memory usage: 187.5 KB


#### 5. Data Type Conversion

In [5]:
# Convert age-group columns from float to int
age_columns = [
    "AGE 25-49",
    "AGE 25-64",
    "AGE 50-64"
]

df[age_columns] = df[age_columns].astype("Int64")

In [6]:
# Remove non-informative column
df = df.drop(columns="REGION")

In [7]:
# Verify dataset structure
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1499 entries, 0 to 1498
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   REGION TYPE        1499 non-null   str    
 1   YEAR               1499 non-null   int64  
 2   WEEK               1499 non-null   int64  
 3   % WEIGHTED ILI     1499 non-null   float64
 4   %UNWEIGHTED ILI    1499 non-null   float64
 5   AGE 0-4            1499 non-null   int64  
 6   AGE 25-49          967 non-null    Int64  
 7   AGE 25-64          627 non-null    Int64  
 8   AGE 5-24           1499 non-null   int64  
 9   AGE 50-64          967 non-null    Int64  
 10  AGE 65             1499 non-null   int64  
 11  ILITOTAL           1499 non-null   int64  
 12  NUM. OF PROVIDERS  1499 non-null   int64  
 13  TOTAL PATIENTS     1499 non-null   int64  
dtypes: Int64(3), float64(2), int64(8), str(1)
memory usage: 180.2 KB


#### 6. Creating a Date Column

In [8]:
# Convert MMWR year/week to date
df["DATE"] = [
    Week(year, week).startdate()
    for year, week in zip(df["YEAR"], df["WEEK"])
]

# Convert to pandas datetime
df["DATE"] = pd.to_datetime(df["DATE"])

In [9]:
# Verify the date column
df[["YEAR", "WEEK", "DATE"]].head()

,YEAR,WEEK,DATE
0,1997,40,1997-09-28
1,1997,41,1997-10-05
2,1997,42,1997-10-12
3,1997,43,1997-10-19
4,1997,44,1997-10-26


#### 7. Reordering Columns

In [10]:
# Reorder columns
columns = (
    ["REGION TYPE", "DATE", "YEAR", "WEEK"] +
    [col for col in df.columns
     if col not in ["REGION TYPE", "DATE", "YEAR", "WEEK"]]
)

df = df[columns]

In [11]:
# Display the first five rows
df.head()

,REGION TYPE,DATE,YEAR,WEEK,% WEIGHTED ILI,%UNWEIGHTED ILI,AGE 0-4,AGE 25-49,AGE 25-64,AGE 5-24,AGE 50-64,AGE 65,ILITOTAL,NUM. OF PROVIDERS,TOTAL PATIENTS
0,National,1997-09-28,1997,40,1.10148,1.21686,179,<NA>,157,205,<NA>,29,570,192,46842
1,National,1997-10-05,1997,41,1.20007,1.28064,199,<NA>,151,242,<NA>,23,615,191,48023
2,National,1997-10-12,1997,42,1.37876,1.23906,228,<NA>,153,266,<NA>,34,681,219,54961
3,National,1997-10-19,1997,43,1.19920,1.14473,188,<NA>,193,236,<NA>,36,653,213,57044
4,National,1997-10-26,1997,44,1.65618,1.26112,217,<NA>,162,280,<NA>,41,700,213,55506


#### 8. Saving the Processed Dataset

In [13]:
# Save the processed dataset
df.to_csv("data/processed/ILINet_processed.csv", index=False)

## Summary

The preprocessing stage prepared the cleaned ILINet dataset for subsequent analysis by:

- converting age-group variables to pandas' nullable integer (`Int64`) data type,
- removing the non-informative `REGION` column,
- creating a `DATE` column from MMWR epidemiological year and week,
- reorganizing the dataset structure for readability, and
- saving the processed dataset for feature engineering and exploratory analysis.

The dataset is now ready for feature engineering.